In [ ]:
import os

os.environ["OPENAI_API_KEY"] = __import__("getpass").getpass("Paste your OpenAI API key: ")

print("API key inserted:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
!pip install openai pandas

In [ ]:
import os
import time
import json
import re
import pandas as pd
from collections import Counter
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Setup complete.")
print("API key found:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
MODEL = "gpt-4o-mini"

def call_openai(prompt, model=MODEL, temperature=0.3, max_tokens=300):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

In [ ]:
import os
import time
import json
import re
import pandas as pd
from collections import Counter
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = __import__("getpass").getpass("Paste your OpenAI API key: ")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Setup complete.")
print("API key found:", bool(os.getenv("OPENAI_API_KEY")))
print("Client created:", client is not None)

In [ ]:
test_prompt = "Say hello in one short sentence."

result = call_openai(test_prompt)

print(result)

STEP 2 SENTIMENT ANALYSIS

In [ ]:
# Initial simple prompt for sentiment analysis

sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

# Test it once
result = call_openai(sentiment_prompt_v1)

print("Sentiment Analysis Result:")
print(result)

PRODUCT DESCRIPTION GENERATION

In [ ]:
# Initial simple prompt for product description

product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

# Test it once
result = call_openai(product_prompt_v1)

print("Product Description Result:")
print(result)

DATA EXTRACTION

In [ ]:
# Initial simple prompt for data extraction
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""
 
# Test it once
result = call_openai(extraction_prompt_v1)
print("Data Extraction Result:")
print(result)

## Part 1, Step 2: Initial Prompt Results

I created three basic zero-shot prompts for sentiment analysis, product description generation, and data extraction. All three prompts produced responses, which confirms that the OpenAI client and helper function are working. However, the prompts do not yet include strict formatting rules, examples, constraints, or validation requirements, so the outputs may be inconsistent across multiple runs.

## Run Prompts 5 Times


In [ ]:
def run_prompt_tests(prompt, runs=5, temperature=0.3, delay=0.5):
    results = []

    for i in range(runs):
        try:
            output = call_openai(prompt, temperature=temperature)
            results.append({
                "run": i + 1,
                "output": output
            })
            print(f"Run {i + 1}/{runs} complete")
            time.sleep(delay)
        except Exception as e:
            results.append({
                "run": i + 1,
                "output": f"ERROR: {e}"
            })

    return pd.DataFrame(results)


def normalize_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def calculate_exact_consistency(df):
    normalized = df["output"].apply(normalize_text)
    counts = Counter(normalized)
    most_common_response, most_common_count = counts.most_common(1)[0]
    consistency = most_common_count / len(df) * 100

    return {
        "total_runs": len(df),
        "unique_outputs": len(counts),
        "most_common_count": most_common_count,
        "consistency_percent": round(consistency, 2),
        "most_common_response": most_common_response
    }


print("✅ Helper functions loaded.")

Run Sentiment five times

In [ ]:
sentiment_v1_5 = run_prompt_tests(sentiment_prompt_v1, runs=5)

display(sentiment_v1_5)

print(calculate_exact_consistency(sentiment_v1_5))

Run Product Description v1 five times

In [ ]:
product_v1_5 = run_prompt_tests(product_prompt_v1, runs=5)

display(product_v1_5)

print(calculate_exact_consistency(product_v1_5))

Run Data Extraction v1 five times

In [ ]:
extraction_v1_5 = run_prompt_tests(extraction_prompt_v1, runs=5)

display(extraction_v1_5)

print(calculate_exact_consistency(extraction_v1_5))

Create a 5-run summary table

In [ ]:
summary_v1_5 = pd.DataFrame([
    {
        "task": "Sentiment Analysis",
        "version": "v1",
        **calculate_exact_consistency(sentiment_v1_5)
    },
    {
        "task": "Product Description",
        "version": "v1",
        **calculate_exact_consistency(product_v1_5)
    },
    {
        "task": "Data Extraction",
        "version": "v1",
        **calculate_exact_consistency(extraction_v1_5)
    }
])

display(summary_v1_5)

## Part 2, Step 3: 5-Run Test Observations

I ran each initial v1 prompt five times to identify early consistency issues. The sentiment analysis prompt generally identified the correct sentiment, but the format could vary because the prompt did not require a specific output style. The product description prompt produced usable descriptions, but the wording, length, structure, and tone varied between runs. The data extraction prompt identified relevant information, but the structure was not guaranteed to be machine-readable or consistent. These early results show why testing a prompt once is not enough for production use.

Run Sentiment 10 times

In [ ]:
sentiment_v1_10 = run_prompt_tests(sentiment_prompt_v1, runs=10)

display(sentiment_v1_10)

print(calculate_exact_consistency(sentiment_v1_10))

Product Description Generation

In [ ]:
product_v1_10 = run_prompt_tests(product_prompt_v1, runs=10)

display(product_v1_10)

print(calculate_exact_consistency(product_v1_10))

data extraction

In [ ]:
extraction_v1_10 = run_prompt_tests(extraction_prompt_v1, runs=10)

display(extraction_v1_10)

print(calculate_exact_consistency(extraction_v1_10))

10-run summary table

In [ ]:
summary_v1_10 = pd.DataFrame([
    {
        "task": "Sentiment Analysis",
        "version": "v1",
        "runs": 10,
        **calculate_exact_consistency(sentiment_v1_10)
    },
    {
        "task": "Product Description",
        "version": "v1",
        "runs": 10,
        **calculate_exact_consistency(product_v1_10)
    },
    {
        "task": "Data Extraction",
        "version": "v1",
        "runs": 10,
        **calculate_exact_consistency(extraction_v1_10)
    }
])

display(summary_v1_10)

## Part 2, Step 4: 10-Run Test Observations

The 10-run test showed that all three v1 prompts had some consistency problems. For the sentiment analysis prompt, there were 6 different outputs in 10 runs. The most common answer only appeared 3 times, so the exact consistency was 30%. This shows that the model usually understood the message, but the prompt was not specific enough about how the answer should look. The product description prompt was the least consistent, with 10 different outputs in 10 runs, giving it 10% consistency. This makes sense because product descriptions are more creative, but it also shows that the prompt needs clearer rules for length, tone, structure, and what details to include. The data extraction prompt did better, with 3 different outputs and 60% consistency, but it still did not give the exact same format every time. Overall, the 10-run test shows that the v1 prompts are too general. They can give useful answers, but they are not consistent enough yet for a reliable customer service chatbot.

15 times

Sentiment

In [ ]:
sentiment_v1_15 = run_prompt_tests(sentiment_prompt_v1, runs=15)

display(sentiment_v1_15)

print(calculate_exact_consistency(sentiment_v1_15))

Product Description

In [ ]:
product_v1_15 = run_prompt_tests(product_prompt_v1, runs=15)

display(product_v1_15)

print(calculate_exact_consistency(product_v1_15))

Data Extraction 

In [ ]:
extraction_v1_15 = run_prompt_tests(extraction_prompt_v1, runs=15)

display(extraction_v1_15)

print(calculate_exact_consistency(extraction_v1_15))

15-run summary table

In [ ]:
summary_v1_15 = pd.DataFrame([
    {
        "task": "Sentiment Analysis",
        "version": "v1",
        "runs": 15,
        **calculate_exact_consistency(sentiment_v1_15)
    },
    {
        "task": "Product Description",
        "version": "v1",
        "runs": 15,
        **calculate_exact_consistency(product_v1_15)
    },
    {
        "task": "Data Extraction",
        "version": "v1",
        "runs": 15,
        **calculate_exact_consistency(extraction_v1_15)
    }
])

display(summary_v1_15)

Create failure analysis table

In [ ]:
failure_analysis_v1 = pd.DataFrame([
    {
        "task": "Sentiment Analysis",
        "prompt_version": "v1",
        "failure_patterns": "The prompt does not define allowed labels or require a specific format. The model may return 'Positive', 'positive', or a longer explanation.",
        "impact": "Inconsistent outputs make it harder to use the result in automated systems.",
        "recommended_improvement": "Add allowed labels and require a single lowercase word with no explanation."
    },
    {
        "task": "Product Description",
        "prompt_version": "v1",
        "failure_patterns": "The prompt does not define length, tone, structure, target audience, or required product details. Outputs may vary widely.",
        "impact": "Brand voice and product page quality may become inconsistent.",
        "recommended_improvement": "Add word count, structure, target customer, tone, and required details."
    },
    {
        "task": "Data Extraction",
        "prompt_version": "v1",
        "failure_patterns": "The prompt does not define specific fields or require machine-readable output. The model may return prose, bullets, or inconsistent field names.",
        "impact": "The output may be difficult to parse or use in CRM/data workflows.",
        "recommended_improvement": "Require valid JSON with exact field names and rules for missing values."
    }
])

display(failure_analysis_v1)

## Part 2, Step 5: 15-Run Failure Analysis

The 15-run test gave a clearer picture of how inconsistent the v1 prompts were. For Sentiment Analysis, the prompt produced 5 unique outputs across 15 runs, and the most common response appeared 7 times, giving it 46.67% exact consistency. This was better than the 10-run test, but it still shows that the model was not following one fixed format. The Product Description prompt was the least consistent again, with 15 unique outputs in 15 runs and only 6.67% exact consistency. This is partly expected because product descriptions are creative, but the prompt still needs clearer instructions for structure, length, tone, and required product details. The Data Extraction prompt produced 3 unique outputs across 15 runs, with the most common response appearing 8 times, giving it 53.33% consistency. This shows that the model usually extracted the right information, but the output format was still not fully standardized. Overall, the 15-run test confirmed that the v1 prompts can produce useful answers, but they are too vague for a reliable production chatbot.

## Failure Analysis Table Summary

The failure analysis table shows the main weaknesses of the v1 prompts. 
The Sentiment Analysis prompt does not define the allowed labels or require a specific output format, 
so the model can respond with different versions of the same answer. The Product Description prompt 
is too open-ended because it does not give rules for length, tone, structure, target audience, or 
required details. This makes the product descriptions inconsistent across runs. 
The Data Extraction prompt has the clearest business risk because it does not require exact fields or 
a machine-readable format, which could make the output difficult to use in a CRM, report, or 
automated workflow. Overall, the table confirms that the v1 prompts need clearer instructions, 
output constraints, and structured formatting.

## PART 3: REWRITING SIMPLE PROMPTS

In [ ]:
sentiment_prompt_v2 = """
You are a customer service sentiment classifier.

Classify the customer message into exactly one of these labels:
positive
negative
neutral

Rules:
- Respond with only one lowercase word.
- Do not explain your answer.
- Do not add punctuation.

Customer message:
"I love this product! It's exactly what I needed."
"""

In [ ]:
result = call_openai(sentiment_prompt_v2)

print("Sentiment Analysis v2 Result:")
print(result)

In [ ]:
sentiment_v2_15 = run_prompt_tests(sentiment_prompt_v2, runs=15)

display(sentiment_v2_15)

print(calculate_exact_consistency(sentiment_v2_15))

Part 3, Step 7: Improve Product Description Prompt

In [ ]:
product_prompt_v2 = """
You are a marketing copywriter for an ecommerce store.

Create a product description for the following product.

Product:
- Product type: Wireless mouse
- Price: $29.99
- Key benefit: Comfortable everyday use
- Target customer: Office workers and students

Output requirements:
- Write 60-80 words.
- Use a friendly, professional tone.
- Make it SEO friendly with relevant keywords.
- Mention the price once.
- Do not use exaggerated claims.
- Do not use bullet points.
- First, include the key benefit addressed to the target customer in the description.
"""

In [ ]:
result = call_openai(product_prompt_v2)

print("Product Description v2 Result:")
print(result)

In [ ]:
product_v2_15 = run_prompt_tests(product_prompt_v2, runs=15)

display(product_v2_15)

print(calculate_exact_consistency(product_v2_15))

Data Extraction

In [ ]:
extraction_prompt_v2 = """
You are a data extraction assistant.

Extract structured information from the customer feedback below.

Return only valid JSON with these exact fields:
{
  "item_number": string or null,
  "order_date": string or null,
  "delivery_sentiment": "positive" or "negative" or "neutral",
  "packaging_issue": true or false,
  "summary": string
}

Rules:
- Use null if a field is not mentioned.
- Do not include markdown.
- Do not include explanation outside the JSON.

Customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

In [ ]:
extraction_prompt_v2 = """
You are a data extraction assistant.

Extract structured information from the customer feedback below.

Return only valid JSON with these exact fields:
{
  "item_number": string or null,
  "order_date": string or null,
  "delivery_sentiment": "positive" or "negative" or "neutral",
  "packaging_issue": true or false,
  "summary": string
}

Rules:
- Use null if a field is not mentioned.
- Do not include markdown.
- Do not include explanation outside the JSON.

Customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

In [ ]:
extraction_v2_15 = run_prompt_tests(extraction_prompt_v2, runs=15)

display(extraction_v2_15)

print(calculate_exact_consistency(extraction_v2_15))

Summary Table

In [ ]:
summary_v2_15 = pd.DataFrame([
    {
        "task": "Sentiment Analysis",
        "version": "v2",
        "runs": 15,
        "consistency_type": "exact output",
        **calculate_exact_consistency(sentiment_v2_15)
    },
    {
        "task": "Product Description",
        "version": "v2",
        "runs": 15,
        "consistency_type": "exact output",
        **calculate_exact_consistency(product_v2_15)
    },
    {
        "task": "Data Extraction",
        "version": "v2",
        "runs": 15,
        "consistency_type": "exact output",
        **calculate_exact_consistency(extraction_v2_15)
    }
])

display(summary_v2_15)

## Part 3: Iteration 1 Results

The v2 prompts showed a clear improvement compared with the v1 prompts. The Sentiment Analysis prompt improved to 100% consistency because the prompt now required one lowercase label from a fixed list. The Data Extraction prompt also improved to 100% consistency because it required valid JSON with exact field names, which made the output much more structured and reliable. The Product Description prompt only improved from 6.67% to 13.33% exact consistency, but this is expected because product descriptions are creative and the wording can change even when the output still follows the instructions. Overall, the v2 prompts were much better than the v1 prompts because they added clearer rules, formatting requirements, and constraints.

Sentiment v1: 46.67% → Sentiment v2: 100%
Product v1: 6.67% → Product v2: 13.33%
Data Extraction v1: 53.33% → Data Extraction v2: 100%

For Product Description, exact consistency is not the best measurement by itself because the model can write different descriptions that still follow the prompt correctly. A better evaluation would check whether each output follows the required word count, mentions the price once, avoids bullet points, uses an appropriate tone, and includes relevant SEO keywords.

PART 4

Add Few-Shot Examples to Sentiment Analysis

In [ ]:
sentiment_prompt_v3 = """
You are a customer service sentiment classifier.

Classify each customer message into exactly one lowercase label:
positive
negative
neutral

Rules:
- Respond with only one lowercase word.
- Do not explain.
- Do not add punctuation.

Examples:

Customer message: "The support team solved my issue quickly. Thank you!"
Sentiment: positive

Customer message: "My order arrived broken and nobody has responded."
Sentiment: negative

Customer message: "I received my package yesterday."
Sentiment: neutral

Now classify this message:

Customer message: "I love this product! It's exactly what I needed."
Sentiment:
"""

In [ ]:
result = call_openai(sentiment_prompt_v3)

print("Sentiment Analysis v3 Result:")
print(result)

In [ ]:
sentiment_v3_15 = run_prompt_tests(sentiment_prompt_v3, runs=15)

display(sentiment_v3_15)

print(calculate_exact_consistency(sentiment_v3_15))

## Part 4, Step 9: Sentiment Analysis v3

For the Sentiment Analysis v3 prompt, I added few-shot examples that show the model exactly how to classify positive, negative, and neutral customer messages. The prompt still keeps the same strict rules from v2, requiring only one lowercase word with no explanation or punctuation. The few-shot examples make the expected output clearer and help the model follow the format more reliably.

Add Chain-of-Thought-style guidance to Data Extraction

In [ ]:
extraction_prompt_v3 = """
You are a careful data extraction assistant.

Extract structured information from customer feedback.

Before giving the final answer, think through:
1. What item number is mentioned?
2. What order date is mentioned?
3. Is the delivery feedback positive, negative, or neutral?
4. Was there a packaging issue?
5. What is the short summary?

Return the final answer using this exact format:

item_number:
order_date:
delivery_sentiment:
packaging_issue:
summary:

Customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

In [ ]:
result = call_openai(extraction_prompt_v3)

print("Data Extraction v3 Result:")
print(result)

In [ ]:
extraction_v3_15 = run_prompt_tests(extraction_prompt_v3, runs=15)

display(extraction_v3_15)

print(calculate_exact_consistency(extraction_v3_15))

Product Description

In [ ]:
product_prompt_v3 = """
You are a professional ecommerce copywriter.

Create a product description using the exact structure shown in the examples.

Rules:
- Output exactly two sections: Headline and Description.
- Headline must be 20-30 words.
- Description must be 60-80 words.
- Use a friendly, professional tone.
- Make it SEO friendly with relevant keywords.
- Mention the price once.
- Do not use exaggerated claims.
- Do not use bullet points.
- Start the description by addressing the key benefit for the target customer.
- Include the key benefit addressed to the target customer in the Headline, must be different than the one in the description.
- The description should answer the: what's in it for me? question for the target customer.
- Include price only in headline. 

Example 1:

Product:
- Product type: Stainless steel water bottle
- Price: $19.99
- Key benefit: Keeps drinks cold during the day
- Target customer: Students and commuters

Output:
Headline: Everyday Hydration That Travels With You

Description: Stay refreshed during classes, commutes, and busy routines with this stainless steel water bottle. Designed for students and commuters, it helps keep drinks cold throughout the day without overcomplicating your schedule. At $19.99, it offers a practical option for everyday hydration with a clean design that fits easily into school bags, backpacks, or work totes.

Example 2:

Product:
- Product type: Desk lamp
- Price: $34.99
- Key benefit: Adjustable brightness for work or study
- Target customer: Remote workers and students

Output:
Headline: Better Lighting For Focused Work Sessions for only 29.99

Description: Work and study more comfortably with this adjustable desk lamp, designed for remote workers and students who need flexible lighting throughout the day. Its adjustable brightness helps support reading, writing, and focused desk tasks without making your setup feel complicated. At $34.99, it is a practical lighting upgrade for home offices, dorm rooms, and study spaces.

Now write the product description.

Product:
- Product type: Wireless mouse
- Price: $29.99
- Key benefit: Comfortable everyday use
- Target customer: Office workers and students

Output:
"""

In [ ]:
result = call_openai(product_prompt_v3)

print("Product Description v3 Result:")
print(result)

In [ ]:
product_v3_15 = run_prompt_tests(product_prompt_v3, runs=15)

display(product_v3_15)

print(calculate_exact_consistency(product_v3_15))

## Part 4, Step 11: Product Description v3 Results

The Product Description v3 prompt used few-shot examples and a clearer structure with a headline and description. Across 15 runs, the exact wording still varied, which is expected for a creative generation task. However, the outputs were more organized because the model had examples to follow and clearer rules for tone, length, SEO keywords, price mention, and structure. This shows that few-shot prompting is useful for improving format consistency even when exact wording changes.

In [ ]:
summary_v3_15 = pd.DataFrame([
    {
        "task": "Sentiment Analysis",
        "version": "v3",
        "runs": 15,
        "consistency_type": "exact output",
        **calculate_exact_consistency(sentiment_v3_15)
    },
    {
        "task": "Product Description",
        "version": "v3",
        "runs": 15,
        "consistency_type": "exact output",
        **calculate_exact_consistency(product_v3_15)
    },
    {
        "task": "Data Extraction",
        "version": "v3",
        "runs": 15,
        "consistency_type": "exact output",
        **calculate_exact_consistency(extraction_v3_15)
    }
])

display(summary_v3_15)

## Part 4: Iteration 2 Results

In this section, I created v3 prompts using few-shot examples and Chain-of-Thought-style guidance. The Sentiment Analysis v3 prompt used examples for positive, negative, and neutral messages, which helped reinforce the expected one-word format. The Data Extraction v3 prompt added a thinking process before the final answer, which helped the model extract the correct fields more consistently. The Product Description v3 prompt used examples and a clearer structure with a headline and description. The product description wording still varied because it is a creative task, but the outputs were more organized and followed the requested structure more closely.

In [ ]:
all_summary_15 = pd.concat([
    summary_v1_15,
    summary_v2_15,
    summary_v3_15
], ignore_index=True)

display(all_summary_15)

In [ ]:
comparison_table = all_summary_15[[
    "task",
    "version",
    "runs",
    "unique_outputs",
    "most_common_count",
    "consistency_percent"
]]

display(comparison_table)

## v1 vs v2 vs v3 Comparison

The comparison table shows how the prompts changed across the three versions. The v1 prompts were the least reliable because they were too general and did not include clear format requirements. The v2 prompts improved consistency by adding stricter instructions, output formats, and constraints. The v3 prompts added few-shot examples and Chain-of-Thought-style guidance, which helped reinforce the expected behavior. Sentiment Analysis and Data Extraction showed the strongest improvements because those tasks benefit from strict labels and structured fields. Product Description remained less exact because creative writing naturally varies, but the structure improved with clearer examples.

Tuning for Different Tasks

In [ ]:
sentiment_variations = [
    "The product works fine, but delivery took longer than expected.",
    "Your support team was amazing and helped me fix everything.",
    "I want a refund. This stopped working after two days.",
    "The package arrived this morning.",
    "I like the design, but the app keeps crashing."
]

def build_sentiment_prompt_v3(message):
    return f"""
You are a customer service sentiment classifier.

Classify each customer message into exactly one lowercase label:
positive
negative
neutral

Rules:
- Respond with only one lowercase word.
- Do not explain.
- Do not add punctuation.

Examples:

Customer message: "The support team solved my issue quickly. Thank you!"
Sentiment: positive

Customer message: "My order arrived broken and nobody has responded."
Sentiment: negative

Customer message: "I received my package yesterday."
Sentiment: neutral

Now classify this message:

Customer message: "{message}"
Sentiment:
"""

sentiment_variation_results = []

for message in sentiment_variations:
    output = call_openai(build_sentiment_prompt_v3(message))
    sentiment_variation_results.append({
        "message": message,
        "sentiment_output": output
    })

sentiment_variation_df = pd.DataFrame(sentiment_variation_results)

display(sentiment_variation_df)

### Sentiment Variation Notes

I tested the final sentiment prompt on different types of customer messages, including positive, negative, neutral, and mixed feedback. The prompt worked best when the customer message had a clear emotional direction. Mixed messages were more challenging because they included both positive and negative details, so the model had to choose the strongest overall sentiment.

Product Description Task Variations

In [ ]:
product_variations = [
    {
        "product_type": "Bluetooth headphones",
        "price": "$49.99",
        "key_benefit": "Clear sound for calls and music",
        "target_customer": "commuters and remote workers"
    },
    {
        "product_type": "Laptop stand",
        "price": "$24.99",
        "key_benefit": "Raises screen height for a better desk setup",
        "target_customer": "students and office workers"
    },
    {
        "product_type": "Portable charger",
        "price": "$39.99",
        "key_benefit": "Keeps devices powered while traveling",
        "target_customer": "travelers and busy professionals"
    }
]

def build_product_prompt_v3(product):
    return f"""
You are a professional ecommerce copywriter.

Create a product description using this exact structure:

Headline:
Description:

Rules:
- Output exactly two sections: Headline and Description.
- Headline must be 6-10 words.
- Description must be 60-80 words.
- Use a friendly, professional tone.
- Make it SEO friendly with relevant keywords.
- Mention the price once.
- Do not use exaggerated claims.
- Do not use bullet points.
- Start the description by addressing the key benefit for the target customer.

Product:
- Product type: {product["product_type"]}
- Price: {product["price"]}
- Key benefit: {product["key_benefit"]}
- Target customer: {product["target_customer"]}

Output:
"""

product_variation_results = []

for product in product_variations:
    output = call_openai(build_product_prompt_v3(product))
    product_variation_results.append({
        "product_type": product["product_type"],
        "price": product["price"],
        "output": output
    })

product_variation_df = pd.DataFrame(product_variation_results)

display(product_variation_df)

### Product Description Variation Notes

I tested the final product description prompt on different ecommerce products. The prompt was able to adapt to different product types, prices, key benefits, and target customers while keeping the same basic structure. The wording changed for each product, which is expected for a creative task, but the headline and description format helped keep the outputs organized.

Data Extraction Task Variations

In [ ]:
extraction_variations = [
    "I ordered item #7788 last Monday. It arrived quickly and everything looked perfect.",
    "My item #ABC22 arrived today, but the box was ripped open.",
    "I placed an order on April 2nd. The delivery was okay, but I never received item #555.",
    "The package was damaged and late. I don't remember the item number.",
    "Item #777 arrived on May 1st. Fast delivery, no issues."
]

def build_extraction_prompt_v3(feedback):
    return f"""
You are a careful data extraction assistant.

Extract structured information from customer feedback.

Before giving the final answer, think through:
1. What item number is mentioned?
2. What order date is mentioned?
3. Is the delivery feedback positive, negative, or neutral?
4. Was there a packaging issue?
5. What is the short summary?

Return the final answer using this exact format:

item_number:
order_date:
delivery_sentiment:
packaging_issue:
summary:

Customer feedback:
"{feedback}"
"""

extraction_variation_results = []

for feedback in extraction_variations:
    output = call_openai(build_extraction_prompt_v3(feedback))
    extraction_variation_results.append({
        "feedback": feedback,
        "output": output
    })

extraction_variation_df = pd.DataFrame(extraction_variation_results)

display(extraction_variation_df)

### Data Extraction Variation Notes

I tested the final data extraction prompt on customer feedback with different details, including missing item numbers, damaged packaging, positive delivery feedback, and mixed delivery issues. The prompt worked best when the feedback clearly included the item number and date. When information was missing, the output still kept the same field structure, which made the result easier to compare across examples.

Final Evaluation and Comparison

In [ ]:
all_summary_15 = pd.concat([
    summary_v1_15,
    summary_v2_15,
    summary_v3_15
], ignore_index=True)

display(all_summary_15)

In [ ]:
final_comparison_table = all_summary_15[[
    "task",
    "version",
    "runs",
    "unique_outputs",
    "most_common_count",
    "consistency_percent"
]]

display(final_comparison_table)

## Step 13: Final Evaluation and Comparison

The final comparison shows that the prompts improved across the three versions. The v1 prompts were the least reliable because they were too general and did not include clear formatting requirements. The v2 prompts improved by adding clearer instructions, constraints, and structured output formats. The v3 prompts added few-shot examples and Chain-of-Thought-style guidance, which helped reinforce the expected behavior. Sentiment Analysis and Data Extraction improved the most because they are structured tasks that benefit from strict labels and fixed fields. Product Description remained less consistent in exact wording because it is a creative task, but the v3 prompt produced more organized responses by using a headline and description format.

In [ ]:
# Create/write a markdown file
summary = """
# Lab Summary

Write your summary here.
"""

with open("lab_summary.md", "w") as file:
    file.write(summary)


In [ ]:
summary_text = """In this lab, I tested and improved prompts for sentiment analysis, product description generation, and data extraction by running each prompt multiple times and comparing the results across v1, v2, and v3. The first version of each prompt worked, but the outputs were inconsistent because the instructions were too general. The second version improved reliability by adding clearer rules, constraints, and structured output formats. The third version used few-shot examples and Chain-of-Thought-style guidance, which helped make the expected behavior clearer. Sentiment Analysis and Data Extraction improved the most because they used fixed labels and structured fields, while Product Description still varied in exact wording because it is a creative task. Next time, I would add simple evaluation checks earlier so I could measure not only exact consistency, but also whether each response followed the required format."""

with open("lab_summary.md", "w") as file:
    file.write(summary_text)

print("✅ lab_summary.md created.")